# CASE expressions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-04-case-expressions.ipynb)

`CASE` is SQL's if/else. It turns values into labels (bucketing), into flags, and, inside an aggregate, into
columns (a pivot). Once you can write `CASE`, most reporting logic fits in one query.

This chapter shows how `CASE` is evaluated, how it works with aggregates, and how Spark's `PIVOT` clause differs.
It ends with practice questions and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

## 1. The syntax

A searched `CASE` lists conditions in order. The result is the value of the **first** condition that is true.

```sql
CASE
  WHEN condition1 THEN value1
  WHEN condition2 THEN value2
  ELSE default_value
END
```

If no condition is true and there is no `ELSE`, the result is `NULL`. The order of the branches is part of the
logic.

In [ ]:
spark.sql("""
SELECT
  CASE WHEN 500 >= 100 THEN 'a' WHEN 500 >= 300 THEN 'b' END AS first_match_wins,
  CASE WHEN 1 = 2 THEN 'x' END                                AS no_else_gives_null
""").show()

## 2. Bucketing values

Label each order as `small`, `medium` or `large`. The branches are checked from top to bottom, so an order of 300
is `large` even though it also satisfies `>= 150`.

In [ ]:
spark.sql("""
SELECT order_id, amount,
       CASE WHEN amount >= 300 THEN 'large'
            WHEN amount >= 150 THEN 'medium'
            ELSE 'small'
       END AS size
FROM orders
ORDER BY order_id
LIMIT 6
""").show()

## 3. CASE inside an aggregate

Put a `CASE` inside `COUNT` or `SUM` and each condition becomes its own column. `COUNT(CASE ... THEN 1 END)` counts
only the matching rows, because a `CASE` with no `ELSE` gives NULL, and `COUNT` ignores NULL. The `LEFT JOIN` keeps
customers with no orders, and they show zeros.

In [ ]:
spark.sql("""
SELECT c.customer_id, c.name,
       COUNT(CASE WHEN o.status = 'completed' THEN 1 END) AS completed,
       COUNT(CASE WHEN o.status = 'cancelled' THEN 1 END) AS cancelled
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.name
ORDER BY c.customer_id
""").show()

## 4. The PIVOT clause

Spark also has a `PIVOT` clause that creates the columns from the values of one column. It is shorter, but two
behaviours matter. Combinations that never occur become NULL, not zero. And a group with no matching rows at all,
such as customers 7 and 8, is missing from the result.

In [ ]:
spark.sql("""
SELECT * FROM (SELECT customer_id, status FROM orders)
PIVOT (COUNT(*) FOR status IN ('completed' AS completed, 'cancelled' AS cancelled))
ORDER BY customer_id
""").show()

## 5. CASE and NULL

A `WHEN` condition that compares with NULL is never true, the same trap as in `WHERE`. Test for missing values with
`IS NULL`.

In [ ]:
spark.sql("""
SELECT
  CASE WHEN CAST(NULL AS INT) = NULL THEN 'match' ELSE 'no match' END AS equals_null,
  CASE WHEN CAST(NULL AS INT) IS NULL THEN 'match' ELSE 'no match' END AS is_null
""").show()

## Summary

- A `CASE` returns the first matching branch. Order the conditions from the most specific to the least specific.
- Without an `ELSE`, an unmatched row gets NULL.
- Conditional aggregation (`COUNT(CASE ...)`, `SUM(CASE ... ELSE 0 END)`) is portable. `PIVOT` is shorter, but it
  produces NULL for missing combinations.

## Practice questions

1. Count the orders in each size label, where `large` means an amount of 300 or more and `small` means less than 300.
   Sort by label.
2. For each customer with orders, show the number of pending orders and the number of completed orders. Sort by `customer_id`.
3. Label each customer `has cancelled` or `no cancelled`, depending on whether they have any cancelled order.
   Show `customer_id` and the label for customers with orders, sorted by `customer_id`.

## Answers

### Question 1

In [ ]:
q = """
SELECT CASE WHEN amount >= 300 THEN 'large' ELSE 'small' END AS size, COUNT(*) AS orders
FROM orders
GROUP BY CASE WHEN amount >= 300 THEN 'large' ELSE 'small' END
ORDER BY size
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("large", 6), ("small", 8)]

### Question 2

In [ ]:
q = """
SELECT customer_id,
       COUNT(CASE WHEN status = 'pending' THEN 1 END)   AS pending,
       COUNT(CASE WHEN status = 'completed' THEN 1 END) AS completed
FROM orders
GROUP BY customer_id
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 0, 3), (2, 0, 2), (3, 0, 2), (4, 1, 1), (5, 0, 2), (6, 0, 1)]

### Question 3

In [ ]:
q = """
SELECT customer_id,
       CASE WHEN COUNT(CASE WHEN status = 'cancelled' THEN 1 END) > 0
            THEN 'has cancelled' ELSE 'no cancelled' END AS label
FROM orders
GROUP BY customer_id
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (1, "no cancelled"), (2, "has cancelled"), (3, "no cancelled"),
    (4, "no cancelled"), (5, "no cancelled"), (6, "has cancelled"),
]